# Zonas y conteo sobre los tracks de FutBot MX

Notebook **exploratorio** (Fase 3 — análisis de eventos). Tomamos como base un
tutorial de zonas de `supervision` y lo reconectamos a los **recursos reales del
proyecto**: en vez de correr un detector genérico sobre un video de tráfico, definimos
regiones en la cancha y contamos los **robots/pelota** que ya trackeó el pipeline
`yolo_sam3 → ByteTrack`.

Aprenderás a:

- Crear **zonas poligonales** para contar objetos por área de la cancha.
- Entender la diferencia entre **presencia instantánea** y **cruces acumulados**.
- Operar las zonas de `supervision` **sobre el JSON de tracks** del proyecto (sin
  volver a invocar el modelo).


## Dos tipos de zonas

**PolygonZone** — como un tapete contador: detecta si un objeto está **dentro** del
polígono en este frame exacto. Mide ocupación instantánea (p. ej. _¿cuántos robots hay
ahora en la media cancha defensiva?_).

**LineZone** — como un torniquete: registra cada vez que un objeto **cruza** la línea
(en cualquier dirección). Acumula a lo largo del video (p. ej. _¿cuántas veces cruzaron
robots la línea de medio campo?_).

```
PolygonZone: ¿cuántos objetos hay AHORA en esta área?
LineZone:    ¿cuántos objetos han CRUZADO esta línea en total?
```


## 0 — Setup y selección del video de reserva

Elegimos un video del **split de reserva** (`split == 0`) — es solo para probar, no
toca testing ni fine-tuning. Todo es config-driven: las rutas salen de
`db_metadata.csv` y del config activo.


In [ ]:
import csv
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv

from src.utils import PROJECT_ROOT, get_abs_path
from src.core.inference_schema import inference_paths

# Pon RUN_HEAVY=True SOLO en el pod (GPU) para producir los tracks con SAM3.
# En False, el notebook espera que el JSON de tracks ya exista (paso 1 ya corrido).
RUN_HEAVY = False
RUN_LABEL = "fase3_eventos"  # namespacing de salidas: outputs/inference/<RUN_LABEL>/...
RESERVE_INDEX = 0  # cuál video de reserva usar (0 = el primero)
MAX_FRAMES = 300  # tope de frames a trackear (videos largos)


def load_active_config():
    """Lee CONFIG_FILENAME del .env y devuelve (nombre, config dict)."""
    env_path = PROJECT_ROOT / ".env"
    cfg_name = None
    for ln in env_path.read_text().splitlines() if env_path.exists() else []:
        ln = ln.strip()
        if ln.startswith("CONFIG_FILENAME") and "=" in ln:
            cfg_name = ln.split("=", 1)[1].strip()
    config = (
        json.loads(get_abs_path(f"configs/{cfg_name}").read_text()) if cfg_name else {}
    )
    return cfg_name, config


def reserve_videos():
    """Rutas (project-relative) de los videos con split == 0, en orden de aparición."""
    csv_path = PROJECT_ROOT / "assets" / "db_metadata.csv"
    with csv_path.open(encoding="utf-8") as fh:
        return [r["ruta"] for r in csv.DictReader(fh) if r["split"] == "0"]


CFG_NAME, CONFIG = load_active_config()
OUTPUTS_DIR = CONFIG.get("working_dirs", {}).get("outputs_dir", "outputs")

VIDEO_REL = reserve_videos()[RESERVE_INDEX]
VIDEO_ABS = get_abs_path(VIDEO_REL)
STEM = Path(VIDEO_REL).stem

# Ruta donde vive (o vivirá) el JSON de tracks de este video bajo el run_label.
TRACKS_JSON, _ = inference_paths(STEM, OUTPUTS_DIR, namespace=RUN_LABEL)

print("config activo:", CFG_NAME)
print("clases:", [c["name"] for c in CONFIG.get("classes", [])])
print("video de reserva:", VIDEO_REL)
print("tracks JSON:", TRACKS_JSON.relative_to(PROJECT_ROOT))

## 1 — Producir los tracks (`yolo_sam3 → ByteTrack`)

Una sola llamada por video vía `run_inference`. Genera el JSON unificado (cabecera +
`frames` + `tracks` con `obj_id` **estables**). **Necesita GPU** (SAM3), así que va
detrás de `RUN_HEAVY`. Si ya lo corriste antes, esta celda se salta y reusamos el JSON
existente.


In [ ]:
from src.core.inference import run_inference

if RUN_HEAVY:
    out = run_inference(
        VIDEO_REL,
        mode="tracking",
        detector="yolo_sam3",
        tracker="bytetrack",
        max_frames=MAX_FRAMES,
        run_label=RUN_LABEL,
        render_video=False,  # aquí solo queremos el JSON; el overlay lo hacemos abajo
        progress=True,
    )
    TRACKS_JSON = Path(out["json"])
    print("tracks ->", TRACKS_JSON.relative_to(PROJECT_ROOT))
else:
    assert TRACKS_JSON.exists(), (
        f"No existe {TRACKS_JSON}. Corre esta celda con RUN_HEAVY=True en el pod, "
        "o apunta TRACKS_JSON a un JSON de tracking ya generado."
    )
    print(
        "[RUN_HEAVY=False] reusando JSON existente:",
        TRACKS_JSON.relative_to(PROJECT_ROOT),
    )

## 2 — Cargar los tracks y reconstruir `Detections` por frame

El JSON de tracks guarda, por `obj_id`, sus observaciones `(frame_index, bbox, centroid,
score)`. Para reaprovechar las zonas de `supervision` reconstruimos, **por frame**, un
`sv.Detections` con `xyxy + class_id + tracker_id`. Así las zonas operan sobre los
tracks reales del proyecto, **sin SAM3** — este paso corre en CPU.

Excluimos `green_floor` del conteo (es el campo, no un agente), igual que el overlay del
proyecto.


In [ ]:
doc = json.loads(TRACKS_JSON.read_text())
CLASS_NAMES = doc["header"]["classes"] if "header" in doc else doc["classes"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
RES = doc.get("resolution") or doc["header"]["resolution"]
H, W = RES["height"], RES["width"]

EXCLUDE = {"green_floor"}  # clases que no son agentes a contar


def build_frame_index(tracks, exclude=EXCLUDE):
    """frame_index -> lista de filas [x1, y1, x2, y2, class_id, obj_id, score]."""
    by_frame = {}
    for t in tracks:
        if t["class"] in exclude:
            continue
        cid = CLASS_TO_ID[t["class"]]
        oid = t["obj_id"]
        for o in t["observations"]:
            x, y, w, h = o["bbox"]
            by_frame.setdefault(o["frame_index"], []).append(
                [x, y, x + w, y + h, cid, oid, o["score"]]
            )
    return by_frame


def detections_for_frame(by_frame, idx):
    """Reconstruye un sv.Detections para el frame idx (vacío si no hay observaciones)."""
    rows = by_frame.get(idx, [])
    if not rows:
        return sv.Detections.empty()
    arr = np.asarray(rows, dtype=float)
    return sv.Detections(
        xyxy=arr[:, :4],
        class_id=arr[:, 4].astype(int),
        tracker_id=arr[:, 5].astype(int),
        confidence=arr[:, 6],
    )


BY_FRAME = build_frame_index(doc["tracks"])
NUM_FRAMES = (max(BY_FRAME) + 1) if BY_FRAME else 0
print(
    f"resolución: {W} x {H} px | frames con tracks: {len(BY_FRAME)} | hasta frame {NUM_FRAMES - 1}"
)
print("clases trackeadas:", [n for n in CLASS_NAMES if n not in EXCLUDE])

## Definir la zona poligonal

El polígono es solo una lista de vértices `(x, y)` en píxeles. Lo definimos **relativo a
la resolución** del video para que funcione en cualquier reserva (apaisado o vertical).
Aquí cubrimos la **media cancha izquierda**.


In [ ]:
# Polígono de la media cancha izquierda (vértices: sup-izq, sup-der, inf-der, inf-izq).
POLYGON_IZQ = np.array([[0, 0], [W // 2, 0], [W // 2, H], [0, H]])

zone = sv.PolygonZone(polygon=POLYGON_IZQ)
zone_annotator = sv.PolygonZoneAnnotator(zone=zone, color=sv.Color.RED, thickness=4)

# Visualizar la zona sobre el primer frame del video real para verificar su posición.
cap = cv2.VideoCapture(str(VIDEO_ABS))
ok, primer_frame = cap.read()
cap.release()
assert ok, f"No pude leer el primer frame de {VIDEO_ABS}"

frame_con_zona = zone_annotator.annotate(scene=primer_frame.copy())
plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(frame_con_zona, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Zona definida — media cancha izquierda (polígono rojo)")
plt.show()

## Procesar el video con conteo por zona

Recorremos los frames del video real, reconstruimos las detecciones desde el JSON,
disparamos la zona y anotamos. **No se invoca el modelo** — solo leemos tracks. El
`PolygonZoneAnnotator` dibuja el polígono y su `current_count` automáticamente.


In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
ID_TO_NAME = {i: n for n, i in CLASS_TO_ID.items()}

TARGET = (
    PROJECT_ROOT / OUTPUTS_DIR / "inference" / RUN_LABEL / STEM / f"{STEM}_zona.mp4"
)
TARGET.parent.mkdir(parents=True, exist_ok=True)

video_info = sv.VideoInfo.from_video_path(str(VIDEO_ABS))
frames_gen = sv.get_video_frames_generator(str(VIDEO_ABS))

with sv.VideoSink(str(TARGET), video_info=video_info) as sink:
    for idx, frame in enumerate(frames_gen):
        if idx >= NUM_FRAMES:
            break
        dets = detections_for_frame(BY_FRAME, idx)
        en_zona = zone.trigger(detections=dets)  # máscara booleana + current_count
        dets_zona = dets[en_zona]

        labels = [
            f"{ID_TO_NAME[c]} #{t}"
            for c, t in zip(dets_zona.class_id, dets_zona.tracker_id)
        ]
        annotated = box_annotator.annotate(scene=frame.copy(), detections=dets_zona)
        annotated = label_annotator.annotate(
            scene=annotated, detections=dets_zona, labels=labels
        )
        annotated = zone_annotator.annotate(scene=annotated)
        sink.write_frame(annotated)

print("Guardado:", TARGET.relative_to(PROJECT_ROOT))

## 🔧 Exploración interactiva


### Experimento 1: Cambiar la posición del polígono


In [ ]:
# El polígono es solo una lista de puntos — cambiarla redefine la zona completamente.
POLYGON_DER = np.array([[W // 2, 0], [W, 0], [W, H], [W // 2, H]])

zone_der = sv.PolygonZone(polygon=POLYGON_DER)
zone_ann_der = sv.PolygonZoneAnnotator(zone=zone_der, color=sv.Color.BLUE, thickness=4)

frame_dcho = zone_ann_der.annotate(scene=primer_frame.copy())
plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(frame_dcho, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Zona media cancha derecha (azul)")
plt.show()
# 💭 Reflexión: ¿en qué media cancha pasan más tiempo los robots en este clip?
# Cambia `zone` por `zone_der` en la celda de procesamiento para comprobarlo.

### Experimento 2: Inspeccionar `current_count` directamente


In [ ]:
# Disparamos la zona sobre un frame concreto para ver qué devuelve trigger().
FRAME_TEST = 0
det_test = detections_for_frame(BY_FRAME, FRAME_TEST)
mascara_zona = zone.trigger(detections=det_test)

print(f"frame {FRAME_TEST} | objetos en el frame: {len(det_test)}")
print(f"máscara de zona (True = dentro): {mascara_zona}")
print(f"objetos dentro de la zona: {int(mascara_zona.sum())}")
print(f"zone.current_count: {zone.current_count}")
# 💭 current_count y mascara_zona.sum() miden lo mismo: ocupación instantánea del polígono.

### Experimento 3: LineZone — cruces de la línea de medio campo

`LineZone` no cuenta presencia instantánea sino **cruces acumulados**. Su API usa
`in_count`/`out_count` en vez de `current_count`. Aquí ponemos una línea vertical en el
medio campo y contamos cuántas veces los robots la cruzan en cada sentido.


In [ ]:
# Línea vertical en el medio campo.
line_zone = sv.LineZone(start=sv.Point(x=W // 2, y=0), end=sv.Point(x=W // 2, y=H))
line_annotator = sv.LineZoneAnnotator(
    thickness=4, text_scale=1.0, custom_in_text="Cruces ->", custom_out_text="Cruces <-"
)

TARGET_LINEA = TARGET.with_name(f"{STEM}_linea.mp4")
frames_gen = sv.get_video_frames_generator(str(VIDEO_ABS))

with sv.VideoSink(str(TARGET_LINEA), video_info=video_info) as sink:
    for idx, frame in enumerate(frames_gen):
        if idx >= NUM_FRAMES:
            break
        dets = detections_for_frame(BY_FRAME, idx)
        line_zone.trigger(detections=dets)  # actualiza in_count / out_count
        annotated = box_annotator.annotate(scene=frame.copy(), detections=dets)
        annotated = line_annotator.annotate(frame=annotated, line_counter=line_zone)
        sink.write_frame(annotated)

print("Guardado:", TARGET_LINEA.relative_to(PROJECT_ROOT))
print(f"Cruces -> (in):  {line_zone.in_count}")
print(f"Cruces <- (out): {line_zone.out_count}")
# 💭 LineZone mide flujo (cuántos pasaron), no ocupación (cuántos hay ahora).
# En fútbol robótico: transiciones ataque/defensa cruzando el medio campo.

## 🚀 Reto de extensión

Muestra la zona poligonal (`PolygonZone`) y la línea de medio campo (`LineZone`) en el
**mismo** video a la vez.

Pista — en el bucle, dispara ambas zonas y aplica los annotators en este orden:

```python
en_zona = zone.trigger(detections=dets)
line_zone.trigger(detections=dets)
annotated = box_annotator.annotate(scene=frame.copy(), detections=dets)
annotated = zone_annotator.annotate(scene=annotated)
annotated = line_annotator.annotate(frame=annotated, line_counter=line_zone)
```

Idea siguiente para la Fase 3: en lugar de zonas geométricas fijas, derivar **eventos**
(posesión, goles, mapas de calor por `obj_id`) directamente de las trayectorias del
JSON.


In [ ]:
# Escribe tu solución aquí.
zone_combo = sv.PolygonZone(polygon=POLYGON_IZQ)
zone_combo_ann = sv.PolygonZoneAnnotator(
    zone=zone_combo, color=sv.Color.RED, thickness=4
)
line_combo = sv.LineZone(start=sv.Point(x=W // 2, y=0), end=sv.Point(x=W // 2, y=H))
line_combo_ann = sv.LineZoneAnnotator(thickness=4, text_scale=1.0)

# ... arma el bucle con sv.VideoSink como en las celdas de arriba ...